In [6]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    _DB_ROOT   = '/content/drive/MyDrive/XAI-Project/DB'
    DRIVE_ROOT = f'{_DB_ROOT}/DB1'
    DATA_PKL   = f'{DRIVE_ROOT}/data_pipeline.pkl'
    MODEL_PKL  = f'{DRIVE_ROOT}/model_config.pkl'
    print('Running in Google Colab — Drive mounted.')
else:
    _ROOT      = os.path.abspath('.')
    DATA_PKL   = os.path.join(_ROOT, 'data_pipeline.pkl')
    MODEL_PKL  = os.path.join(_ROOT, 'model_config.pkl')
    print('Running locally (VS Code / Jupyter).')

print(f'DATA_PKL  : {DATA_PKL}')
print(f'MODEL_PKL : {MODEL_PKL}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running in Google Colab — Drive mounted.
DATA_PKL  : /content/drive/MyDrive/XAI-Project/DB/DB1/data_pipeline.pkl
MODEL_PKL : /content/drive/MyDrive/XAI-Project/DB/DB1/model_config.pkl


## 1. Imports
Import the necessary PyTorch modules and pretrained model weights.

In [7]:
import pickle
import numpy as np
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

## 2. Model Definition
Define the `MultiLevelCBM` class with a ResNet-50 backbone and three output heads.

In [8]:
class HierarchicalCBM(nn.Module):
    """
    Hierarchical Concept Bottleneck Model (H-CBM).

    Architecture:
        Backbone    : ResNet-50 (pretrained ImageNet) → Feature Map F (2048,)
        Coarse Head : g_c(F)          → p_c (NUM_L1,)   — coarse part probabilities
        Fine Head   : g_f(F, p_c)     → p_f (NUM_L2,)   — fine attribute probabilities
                      with Masked Fine Head: p_f[i] = σ(z_f[i]) × p_c[parent(i)]
        Classifier  : h(p_f)          → (num_classes,)  — reads ONLY from p_f

    Key properties:
        - Bottleneck: classifier never sees raw features — only concepts
        - Masked Fine Head: hard architectural hierarchy constraint
        - attr_parent_idx: maps each L2 attribute to its L1 parent (-1 = no parent)
    """

    def __init__(
        self,
        attr_parent_idx: np.ndarray,
        num_classes: int = 200,
        num_l1: int = 13,
        num_l2: int = 312,
    ):
        super().__init__()

        self.num_l1 = num_l1
        self.num_l2 = num_l2

        # parent index per L2 attribute — used in Masked Fine Head
        # -1 means no L1 parent (always unmasked)
        self.register_buffer(
            'attr_parent_idx',
            torch.tensor(attr_parent_idx, dtype=torch.long)
        )

        # ── Backbone ──────────────────────────────────────────────────────────
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        # output: (B, 2048, 1, 1) → flatten → (B, 2048)

        # ── Coarse Head g_c(F) ────────────────────────────────────────────────
        self.coarse_head = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_l1),
        )

        # ── Fine Head g_f(F, p_c) ─────────────────────────────────────────────
        # input = concat(F, p_c) = 2048 + num_l1 = 2061
        self.fine_head = nn.Sequential(
            nn.Linear(2048 + num_l1, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_l2),
        )

        # ── Classifier h(p_f) ─────────────────────────────────────────────────
        # reads ONLY from p_f — never from raw features
        self.classifier = nn.Linear(num_l2, num_classes)

    def forward(self, x: torch.Tensor):
        """
        Args:
            x : (B, 3, 224, 224) — batch of RGB images

        Returns:
            cls_logits : (B, 200)    — species logits
            p_c        : (B, 13)     — coarse concept probabilities
            p_f        : (B, 312)    — fine attribute probabilities (masked)
        """
        # ── Backbone ──────────────────────────────────────────────────────────
        feats = self.features(x).flatten(1)   # (B, 2048)

        # ── Coarse Head ───────────────────────────────────────────────────────
        z_c = self.coarse_head(feats)          # (B, 13) logits
        p_c = torch.sigmoid(z_c)              # (B, 13) probabilities

        # ── Fine Head ─────────────────────────────────────────────────────────
        fine_input = torch.cat([feats, p_c], dim=1)  # (B, 2061)
        z_f        = self.fine_head(fine_input)       # (B, 312) logits
        p_f_raw    = torch.sigmoid(z_f)              # (B, 312) raw probabilities

        # ── Masked Fine Head ──────────────────────────────────────────────────
        # p_f[i] = p_f_raw[i] × p_c[parent(i)]
        # attributes with no parent (idx == -1) are always unmasked (× 1.0)
        parent_idx   = self.attr_parent_idx          # (312,)
        has_parent   = parent_idx >= 0               # (312,) bool mask

        # gather p_c values for each attribute's parent
        # for no-parent attributes, use index 0 temporarily (will be overridden)
        safe_idx     = parent_idx.clamp(min=0)       # (312,)
        parent_probs = p_c[:, safe_idx]              # (B, 312)

        # attributes without a parent are always unmasked
        mask         = torch.where(
            has_parent.unsqueeze(0),                 # (1, 312)
            parent_probs,                            # use parent probability
            torch.ones_like(parent_probs),           # no parent → mask = 1
        )
        p_f = p_f_raw * mask                         # (B, 312) — masked

        # ── Classifier ────────────────────────────────────────────────────────
        cls_logits = self.classifier(p_f)            # (B, 200)

        return cls_logits, p_c, p_f


print("HierarchicalCBM class defined.")

HierarchicalCBM class defined.


## 3. Instantiate and Inspect the Model
Create a model instance and print the total number of trainable parameters.

In [9]:
import pickle

# Load metadata saved by data.ipynb
with open(DATA_PKL, 'rb') as f:
    pipeline = pickle.load(f)

NUM_L1          = pipeline['NUM_L1']           # 13
NUM_L2          = pipeline['NUM_L2']           # 312
CONCEPT_NAMES   = pipeline['CONCEPT_NAMES']
attr_parent_idx = pipeline['attr_parent_idx']  # (312,) — for Masked Fine Head

print(f'NUM_L1         : {NUM_L1}')
print(f'NUM_L2         : {NUM_L2}')
print(f'CONCEPT_NAMES  : {CONCEPT_NAMES}')
print(f'attr_parent_idx: shape={attr_parent_idx.shape}, sample={attr_parent_idx[:5]}')

# Instantiate model
model = HierarchicalCBM(
    attr_parent_idx = attr_parent_idx,
    num_classes     = 200,
    num_l1          = NUM_L1,
    num_l2          = NUM_L2,
)

# Count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\nTotal parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')
print(f'\n── Head shapes ──────────────────────────────')
print(f'  Backbone output : 2048')
print(f'  Coarse Head     : 2048 → 512 → {NUM_L1}')
print(f'  Fine Head       : {2048+NUM_L1} → 512 → {NUM_L2}')
print(f'  Classifier      : {NUM_L2} → 200  (reads only from p_f)')


NUM_L1         : 13
NUM_L2         : 312
CONCEPT_NAMES  : ['back', 'belly', 'bill', 'breast', 'crown', 'eye', 'forehead', 'head', 'leg', 'nape', 'tail', 'throat', 'wing']
attr_parent_idx: shape=(312,), sample=[2 2 2 2 2]
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 161MB/s] 



Total parameters     : 25,842,189
Trainable parameters : 25,842,189

── Head shapes ──────────────────────────────
  Backbone output : 2048
  Coarse Head     : 2048 → 512 → 13
  Fine Head       : 2061 → 512 → 312
  Classifier      : 312 → 200  (reads only from p_f)


## 4. Verify Forward Pass
Send a dummy batch through the model to confirm output shapes are correct before connecting the real DataLoader.

In [10]:
model.eval()

dummy_input = torch.randn(4, 3, 224, 224)

with torch.no_grad():
    cls_out, p_c_out, p_f_out = model(dummy_input)

print('Forward pass successful!')
print(f'  cls_logits : {tuple(cls_out.shape)}    ← (batch, 200 species)')
print(f'  p_c        : {tuple(p_c_out.shape)}     ← (batch, {NUM_L1} coarse parts)')
print(f'  p_f        : {tuple(p_f_out.shape)}   ← (batch, {NUM_L2} fine attributes)')

# Sanity check — Masked Fine Head
# If mask works correctly, p_f should not be larger than its parent's probability
violations = 0
for attr_idx in range(NUM_L2):
    parent = attr_parent_idx[attr_idx]
    if parent >= 0:
        child_max  = p_f_out[:, attr_idx].max().item()
        parent_max = p_c_out[:, parent].max().item()
        if child_max > parent_max + 1e-4:
            violations += 1

print(f'\nMasked Fine Head check:')
print(f'  Hierarchy violations: {violations} / {NUM_L2}')
print(f'  {"✅ OK" if violations == 0 else "❌ Fix needed"}')

Forward pass successful!
  cls_logits : (4, 200)    ← (batch, 200 species)
  p_c        : (4, 13)     ← (batch, 13 coarse parts)
  p_f        : (4, 312)   ← (batch, 312 fine attributes)

Masked Fine Head check:
  Hierarchy violations: 0 / 312
  ✅ OK


## 5. Select Device
Move the model to GPU if available, otherwise fall back to CPU.

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)

print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
else:
    print('Running on CPU — OK for model definition and forward pass check')
    print('GPU needed for training')

Device : cuda
GPU    : Tesla T4


## 6. Save model architecture info for train.ipynb


In [12]:
import pickle

model_config = {
    'num_classes':     200,
    'num_l1':          NUM_L1,
    'num_l2':          NUM_L2,
    'attr_parent_idx': attr_parent_idx,
    'CONCEPT_NAMES':   CONCEPT_NAMES,
}

with open(MODEL_PKL, 'wb') as f:
    pickle.dump(model_config, f)

print(f'Model config saved to {MODEL_PKL}')


Model config saved to /content/drive/MyDrive/XAI-Project/DB/DB1/model_config.pkl
